# Module E: independent evaluation (Exp 1 accuracy vs K, equivariance probe, Exp 3 noise robustness, Exp 4 trap ablation)
Loads the planners that the **Module B** notebook saved (it trains nothing except the independent evaluator `E_main`, once). E sees only random-dot images during training, never planner output.

**Nothing repeats:** data, layout, planners and evaluator stay in memory; the evaluator is saved to `OUT/E_main_s*.pt`; each experiment's result is saved as json in `OUT` and loaded on later runs. Set `RERUN_EXPS = True` to recompute the experiments, `RETRAIN_EVALUATOR = True` to retrain E_main (both reset themselves after use).

Run order for the whole project: A (independent), C, B, then this notebook.

In [ ]:
import os, sys, json, math, time, subprocess, urllib.parse, urllib.request
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image, ImageDraw
import yaml, matplotlib.pyplot as plt
from scipy import ndimage
try: import dynaphos
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "dynaphos"]); import dynaphos
from dynaphos.simulator import GaussianSimulator
from dynaphos.utils import Map
from dynaphos.cortex_models import get_cortex_coordinates_grid, get_visual_field_coordinates_from_cortex_full

# ---- keep-in-memory helper: an object is built once per kernel; re-running a cell re-uses it ----
_mem = globals().setdefault("_mem", {})
def once(name, key, build, verbose=True):
    if name in globals() and _mem.get(name) == key:
        if verbose: print(f"[memory] {name}: already loaded, nothing recomputed")
        return globals()[name]
    globals()[name] = build(); _mem[name] = key; return globals()[name]


In [ ]:
# ======== shared settings: keep IDENTICAL in the notebooks for modules B, C and E ========
# (they find each other's files through the OUT folder, whose name is built from these values)
QUICK   = True        # True: small end-to-end run (minutes). False: full run (use a GPU)
SOURCE  = "shapes"    # "shapes" = circle/square/triangle/line (plan step A1) | "quickdraw" = stroke drawings
CLASSES = ["circle", "square", "triangle", "line"]    # quickdraw example: ["apple", "bed", "cup", "house", "key", "scissors"]
SIZE, RES, FOV = 256, 256, 10.0     # image size, Dynaphos render size, field of view in degrees
N_ELEC  = 500         # electrodes used (subsampled from a cortical grid inside the field)
I_MAX   = 200.0       # uA. MY CHOICE: brightness ~0.95 here with Dynaphos defaults. Verify against the safety literature.
SEED    = 0           # one seed per run; set 1, 2 and re-run for the other seeds (each seed is cached separately)
N_IMG   = dict(train=60, val=20, test=30) if QUICK else dict(train=400, val=80, test=100)   # drawings per class
dev = "cuda" if torch.cuda.is_available() else "cpu"
NC  = len(CLASSES)
OUT = f"pipeline_out_{'quick' if QUICK else 'full'}_{SOURCE}_{NC}c_{N_ELEC}e_{RES}"; os.makedirs(OUT, exist_ok=True)
KEY = (SEED, N_ELEC, SOURCE, tuple(CLASSES), QUICK, RES, FOV, I_MAX)      # anything cached in memory is rebuilt only if this changes
QD_URL = "https://storage.googleapis.com/quickdraw_dataset/full/simplified/{}.ndjson"
print("device:", dev, "| outputs in", OUT)

# ======== experiment settings (modules B and E) ========
K_LIST = [5, 16, 32] if QUICK else [3, 5, 8, 12, 16, 24, 32, 48, 64]    # K grid for the accuracy-vs-K sweep
K_B2B  = [16] if QUICK else [8, 16, 32]                                  # K values for which the learned planners are trained
KB     = K_B2B[len(K_B2B) // 2]                                          # the K used for ablations / noise / probe
STEPS, BS, EPOCHS = (80, 8, 4) if QUICK else (600, 16, 15)               # B2b steps, batch size, classifier epochs
N_REP  = 2 if QUICK else 4                                               # random-dot samples per training image (T_c and E)
BOOT   = 500 if QUICK else 5000                                          # bootstrap resamples
VARIANTS = {"both": (1., .1, False), "contour_only": (1., 0., False), "task_only": (0., 1., False), "robust": (1., .1, True)}   # (lambda_contour, lambda_task, perturbation training)
def b2a_path(K): return f"{OUT}/B2a_K{K}_s{SEED}.npy"
def b2b_path(v, K): return f"{OUT}/B2b_{v}_K{K}_s{SEED}.pt"


In [ ]:
# ---------------------------------------------------------------- Module C: frozen Dynaphos + electrode atlas
PARAMS_YAML = f"""
run: {{resolution: [{RES}, {RES}], view_angle: {FOV}, origin: [0, 0], min_angle: 0.001, fps: 35,
      gpu: {0 if torch.cuda.is_available() else 'null'}, print_stats: false, seed: 42, dtype: float32,
      use_gaussian_lut: false, batch_size: 0}}
display: {{screen_resolution: [1920, 1080], screen_diagonal: 13.3, dist_to_screen: 600}}
sampling: {{sampling_method: receptive_fields, RF_size: 0.5, stimulus_scale: 1.e-4}}
cortex_model: {{model: dipole, k: 17.3, a: 0.75, b: 120, alpha: 0.95, dropout_rate: 0.0, noise_scale: 0.4}}
temporal_dynamics: {{trace_increase_rate: 13.95528162, activation_decay_per_second: 0.00012340980408667956, trace_decay_per_second: 0.99949191}}
size: {{size_equation: sqrt, MD: 0.7, I_half: 40, slope_size: 0.08, current_spread: 675.e-6, radius_to_sigma: 0.5}}
brightness_saturation: {{cps_half: 1.057631326853325e-07, slope_brightness: 19152642.500946816}}
thresholding: {{use_threshold: true, rheobase: 23.9e-6, activation_threshold: 9.141886000943878e-08, activation_threshold_sd: 6.715877869833961e-08}}
default_stim: {{relative_stim_duration: 1, pw_default: 170.e-6, freq_default: 300}}
gabor: {{gabor_filtering: false, gamma: 1.5}}
"""   # = dynaphos config/params.yaml; changed: resolution, view_angle, dropout_rate 0 (dropout is applied at render time). Source: Dynaphos repo defaults.

class Phos:
    """Frozen Dynaphos + electrode atlas. render(I_uA) -> phosphene images; only active electrodes are summed (same maths as sim.__call__)."""
    def __init__(s, n_elec, seed):
        s.P = yaml.safe_load(PARAMS_YAML); rng = np.random.default_rng(seed)
        g = 32 if n_elec <= 1000 else 48
        cc = get_cortex_coordinates_grid(s.P["cortex_model"], g, g, x_max=40)       # g x g electrodes per hemisphere on the cortex
        x, y = get_visual_field_coordinates_from_cortex_full(s.P["cortex_model"], cc, rng).cartesian
        inb = (np.abs(x) < FOV / 2) & (np.abs(y) < FOV / 2)
        s.n_total, s.n_in_field = len(x), int(inb.sum())
        keep = rng.choice(np.flatnonzero(inb), min(n_elec, s.n_in_field), replace=False)
        s.sim = GaussianSimulator(s.P, Map(x=x[keep], y=y[keep]), rng)
        s.N, s.dev = s.sim.num_phosphenes, s.sim.data_kwargs["device"]
        co = s.sim.phosphene_maps.flatten(1).argmin(1).cpu().numpy()                 # pixel index of each phosphene centre
        s.xy = np.c_[x[keep], y[keep]]; s.px = np.c_[co % RES, co // RES]            # visual-field (deg) and pixel (col,row) centres
        p, dp, st = s.P["thresholding"], s.P["default_stim"], s.P["run"]["fps"]
        s.k_cur = st / (dp["pw_default"] * dp["freq_default"] * dp["relative_stim_duration"])   # activation -> current factor
        s.thr0 = s.sim.threshold.get().flatten()
        s.thr = ((p["rheobase"] + s.thr0.cpu().numpy() * s.k_cur) * 1e6 * 1.001 + 1e-3).astype(np.float32)   # uA, lowest current that shows a phosphene
        s.usable = s.thr <= I_MAX                                                    # electrodes that can ever be driven
        s.sim.reset(); s.sim.update(torch.full((s.N,), 128e-6, device=s.dev))        # size atlas at a reference current of 128 uA
        s.sigma_deg = s.sim.sigma.get().flatten().cpu().numpy(); s.sigma_px = s.sigma_deg / FOV * (RES - 1)
        s.ecc = np.linalg.norm(s.xy, axis=1)
        b = s.P["brightness_saturation"]; lvl = .7
        s.i70 = float((p["rheobase"] + (b["cps_half"] + math.log(lvl / (1 - lvl)) / b["slope_brightness"]) * s.k_cur) * 1e6)   # uA for 70 % brightness
        sg = torch.tensor(np.maximum(s.sigma_deg, 1.0 * FOV / (RES - 1)), device=s.dev)[:, None, None]
        W = torch.exp(-.5 * (s.sim.phosphene_maps / sg) ** 2).flatten(1); s.W = W / W.sum(1, keepdim=True)   # Gaussian sampling weights per electrode

    def render(s, I, p_drop=0., sigma_thr=0.):
        """I: (B,N) uA tensor (differentiable) -> (B,1,RES,RES). Optional electrode dropout and log-normal threshold jitter."""
        k = max(int((I > 0).sum(1).max()), 1); idx = I.topk(k, 1).indices; out = []
        for b in range(len(I)):
            if sigma_thr > 0: s.sim.threshold.state = (s.thr0 * torch.exp(sigma_thr * torch.randn_like(s.thr0))).view(-1, 1, 1)
            s.sim.reset(); s.sim.update(I[b].clamp_min(1e-3) * 1e-6)                 # clamp: sqrt(0) has an infinite gradient
            i = idx[b]; sg = s.sim.sigma.get().flatten()[i]; br = s.sim.brightness.get().flatten()[i]
            on = (s.sim.activation.get().flatten() > s.sim.threshold.get().flatten())[i].float() * (I[b][i] > 0).float()
            if p_drop > 0: on = on * (torch.rand(k, device=I.device) > p_drop).float()
            g = torch.exp(-.5 * (s.sim.phosphene_maps[i] / sg.clamp_min(1e-22)[:, None, None]) ** 2)
            out.append((on * br)[:, None, None].mul(g).sum(0))
        s.sim.threshold.state = s.thr0.view(-1, 1, 1)
        return torch.stack(out).clamp(0, 1)[:, None]

def save_atlas(C):
    np.savez(f"{OUT}/atlas_s{SEED}.npz", xy_deg=C.xy, px=C.px, sigma_deg=C.sigma_deg, thr_uA=C.thr, usable=C.usable, i70_uA=C.i70)

def check_atlas(C):
    f = f"{OUT}/atlas_s{SEED}.npz"
    if not os.path.exists(f): print("note: no atlas from the Module C notebook yet (run it once). The layout is rebuilt here from the same settings."); return
    a = np.load(f)
    assert a["xy_deg"].shape == C.xy.shape and np.allclose(a["xy_deg"], C.xy) and np.allclose(a["thr_uA"], C.thr, rtol=1e-3), \
        "this layout differs from the atlas written by the Module C notebook: settings or seed changed. Re-run Module C."
    print("layout matches the atlas written by the Module C notebook")


In [ ]:
# ---------------------------------------------------------------- data: stroke drawings -> 256x256 images, contour targets (cached on disk)
def shape_strokes(c, rng):
    n = {"circle": 48, "square": 4, "triangle": 3, "line": 2}[c]
    a = np.linspace(0, 2 * np.pi, n, endpoint=False) + (np.pi / 4 if c == "square" else 0)
    p = np.c_[np.cos(a), np.sin(a) * (rng.uniform(.6, 1) if c != "line" else 0)]
    if c != "line": p = np.vstack([p, p[:1]])                       # close the outline
    t = rng.uniform(0, 2 * np.pi); p = p @ np.array([[np.cos(t), -np.sin(t)], [np.sin(t), np.cos(t)]])
    return [[list(p[:, 0]), list(p[:, 1])]]

def qd_strokes(name, n):
    f = f"{OUT}/qd_{name}.json"
    if os.path.exists(f) and len(json.load(open(f))) >= n: return json.load(open(f))[:n]
    req = urllib.request.Request(QD_URL.format(urllib.parse.quote(name)), headers={"Range": f"bytes=0-{3000 * n}"})
    raw = urllib.request.urlopen(req, timeout=60).read(3000 * n + 1).decode("utf-8", "ignore").split("\n")[:-1]   # last line may be cut
    out = [d["drawing"] for d in map(json.loads, raw) if d["recognized"]]
    if len(out) < n: raise RuntimeError(f"only {len(out)} drawings for {name}; raise the byte range")
    json.dump(out, open(f, "w")); return out[:n]

def draw(strokes, rng):
    pts = np.concatenate([np.c_[s[0], s[1]] for s in strokes]).astype(float)
    lo, hi = pts.min(0), pts.max(0); span = max((hi - lo).max(), 1e-6)
    ext = rng.uniform(.5, .95) * SIZE                                # object spans 50-95 % of the field
    ctr = SIZE / 2 + rng.uniform(-1, 1, 2) * (SIZE - ext) / 2        # position jitter
    im = Image.new("L", (SIZE, SIZE), 0); d = ImageDraw.Draw(im)
    for s in strokes:
        p = (np.c_[s[0], s[1]] - (lo + hi) / 2) * ext / span + ctr
        if len(p) > 1: d.line(list(map(tuple, p)), fill=255, width=3, joint="curve")
    return np.asarray(im)

def make_split(split, seed):
    off = {"train": 0, "val": N_IMG["train"], "test": N_IMG["train"] + N_IMG["val"]}[split]; n = N_IMG[split]
    X, y = [], []
    for ci, c in enumerate(CLASSES):
        qd = qd_strokes(c, sum(N_IMG.values())) if SOURCE == "quickdraw" else None
        for i in range(n):
            rng = np.random.default_rng([seed, off + i, ci])
            X.append(draw(qd[off + i] if qd else shape_strokes(c, rng), rng)); y.append(ci)
    return np.stack(X), np.array(y)

def gabor_bank(sig=2., lam=4., n=8, ks=15):
    r = torch.arange(ks, dtype=torch.float32) - ks // 2; yy, xx = torch.meshgrid(r, r, indexing="ij"); g = torch.exp(-(xx ** 2 + yy ** 2) / (2 * sig ** 2)); w = []
    for th in np.arange(n) * np.pi / n:
        ph = 2 * np.pi * (xx * np.cos(th) + yy * np.sin(th)) / lam
        ev = g * torch.cos(ph); ev = ev - g * ev.sum() / g.sum()                       # even part with the DC removed
        w += [ev, g * torch.sin(ph)]
    return torch.stack(w)[:, None].to(dev)                                            # (16,1,ks,ks): 8 orientations x (even, odd)

def blur(x, sig):
    r = int(3 * sig) + 1; k = torch.exp(-.5 * (torch.arange(-r, r + 1, device=x.device) / sig) ** 2); k = (k / k.sum()).to(x.dtype)
    return F.conv2d(F.conv2d(x, k.view(1, 1, 1, -1), padding=(0, r)), k.view(1, 1, -1, 1), padding=(r, 0))

def contour_map(x, GB):                                                               # x (B,1,256,256) -> C = sqrt(even^2 + odd^2), max over orientation
    r = F.conv2d(x, GB, padding=GB.shape[-1] // 2).view(len(x), 8, 2, SIZE, SIZE)
    c = r.pow(2).sum(2).sqrt().amax(1, keepdim=True); return c / c.amax((2, 3), keepdim=True).clamp_min(1e-6)

def prep_split(C, X, y, GB):
    """raw uint8 images (n,256,256) -> dict(x image, T blurred contour target, C contour map, score per electrode, y, X)."""
    o = {k: [] for k in ("x", "T", "C", "score")}; r = SIZE // RES
    for i in range(0, len(X), 16):
        x = torch.tensor(X[i:i + 16], dtype=torch.float32, device=dev)[:, None] / 255
        c = F.avg_pool2d(contour_map(x, GB), r); c = c / c.amax((2, 3), keepdim=True).clamp_min(1e-6)
        T = blur(c, 1.5); T = T / T.amax((2, 3), keepdim=True).clamp_min(1e-6)       # blurred contour target
        o["x"].append(F.avg_pool2d(x, r)); o["T"].append(T); o["C"].append(c[:, 0].cpu().numpy()); o["score"].append((c.flatten(1) @ C.W.T).cpu().numpy())
    D = {k: (np.concatenate(v) if k in ("C", "score") else torch.cat(v)) for k, v in o.items()}; D["y"] = y; D["X"] = X; return D

def get_data(C, GB, rebuild=False):
    """Builds the shapes + contour targets ONCE, saves them to OUT/data_s{SEED}.npz, and always loads from that file afterwards."""
    f = f"{OUT}/data_s{SEED}.npz"
    if rebuild or not os.path.exists(f):
        t0 = time.time(); z = {}
        for i, s in enumerate(("train", "val", "test")):
            D = prep_split(C, *make_split(s, 1000 * SEED + i), GB)
            z.update({f"{s}_X": D["X"], f"{s}_y": D["y"], f"{s}_T": D["T"].cpu().numpy().astype(np.float16),
                      f"{s}_C": D["C"].astype(np.float16), f"{s}_score": D["score"].astype(np.float32)})
        np.savez_compressed(f, **z); print(f"data: built and saved to {f} ({time.time() - t0:.0f}s)")
    z = np.load(f); D = {}
    for s in ("train", "val", "test"):
        X = z[f"{s}_X"]
        D[s] = dict(X=X, y=z[f"{s}_y"], x=F.avg_pool2d(torch.tensor(X, dtype=torch.float32, device=dev)[:, None] / 255, SIZE // RES),
                    T=torch.tensor(z[f"{s}_T"].astype(np.float32), device=dev), C=z[f"{s}_C"].astype(np.float32), score=z[f"{s}_score"])
    print(f"data: loaded from {f} | {NC} classes {CLASSES} | images per class train/val/test {N_IMG['train']}/{N_IMG['val']}/{N_IMG['test']}"); return D


In [ ]:
# ---------------------------------------------------------------- reference arms, B1, planner network, constraints
def pick(xy, score, K, n0=120, r0=.15, floor=.1):
    """Greedy best-score selection with exclusion radius r0 (deg) and a 10 % floor, then homogeneous thinning to K (paper's density reduction)."""
    if score.max() <= 0: return np.array([], int)
    s = np.where(score >= floor * score.max(), score, -1.); ch = []
    while len(ch) < n0 and s.max() > 0:
        e = int(s.argmax()); ch.append(e); s[np.linalg.norm(xy - xy[e], axis=1) < r0] = -1
    ch = np.array(ch); Dm = np.linalg.norm(xy[ch][:, None] - xy[ch][None], axis=-1); alive = np.ones(len(ch), bool)
    while alive.sum() > K:
        i, j = np.unravel_index(np.where(alive[:, None] & alive[None] & ~np.eye(len(ch), dtype=bool), Dm, np.inf).argmin(), Dm.shape)
        alive[i if score[ch[i]] < score[ch[j]] else j] = False                        # drop the weaker of the closest pair
    return ch[alive]

def plan_b1(C, score, K):
    ch = pick(C.xy, score * C.usable, K); I = np.zeros(C.N, np.float32); I[ch] = np.clip(C.i70, C.thr[ch], I_MAX); return I

def plan_r1(C, K, rng):
    ch = rng.choice(np.flatnonzero(C.usable), K, replace=False); I = np.zeros(C.N, np.float32); I[ch] = rng.uniform(C.thr[ch], I_MAX); return I

def dots(pts, sig=1.5, amp=.9):                                                       # pts (K,2) pixel (col,row) -> Gaussian dot image (idealised scoreboard)
    g = np.arange(RES); d2 = (g[None, :, None] - pts[:, 0, None, None]) ** 2 + (g[None, None, :] - pts[:, 1, None, None]) ** 2
    return np.exp(-.5 * d2 / sig ** 2).sum(0).clip(0, 1).T * amp if len(pts) else np.zeros((RES, RES))

def plan_r0(Cmap, K, rng, thin=True):
    ys, xs = np.nonzero(Cmap > .2); sel = rng.choice(len(xs), min(600, len(xs)), replace=False)
    pts = np.c_[xs[sel], ys[sel]].astype(float); sc = Cmap[ys[sel], xs[sel]]
    ch = pick(pts * FOV / (RES - 1), sc, K) if thin else rng.choice(len(pts), min(K, len(pts)), replace=False)
    return dots(pts[ch])

def topk_st(logits, K):                                                               # hard top-K forward, sigmoid gradient backward
    soft = torch.sigmoid(logits); hard = torch.zeros_like(soft).scatter_(1, logits.topk(K, 1).indices, 1.)
    return hard + soft - soft.detach()

def currents(C, logits, u, K):                                                        # I = m * (thr + (I_max - thr) * sigmoid(u)), always in [thr, I_max]
    thr = torch.tensor(C.thr, device=dev); lg = logits.masked_fill(~torch.tensor(C.usable, device=dev), -1e4)
    return topk_st(lg, K) * (thr + (I_MAX - thr) * torch.sigmoid(u))

class Planner(nn.Module):
    """B2b: small CNN encoder -> features sampled at each electrode's phosphene centre -> shared MLP -> (selection logit, amplitude logit)."""
    def __init__(s, C):
        super().__init__()
        cv = lambda i, o, d: nn.Sequential(nn.Conv2d(i, o, 3, padding=d, dilation=d), nn.ReLU())
        s.enc = nn.Sequential(cv(1, 16, 1), cv(16, 16, 2), cv(16, 16, 4), cv(16, 16, 8), nn.Conv2d(16, 16, 1))
        s.mlp = nn.Sequential(nn.Linear(18, 64), nn.ReLU(), nn.Linear(64, 2))
        s.register_buffer("grid", torch.tensor(C.px / (RES - 1) * 2 - 1, dtype=torch.float32)[None, :, None])
        s.register_buffer("geo", torch.tensor(np.c_[C.sigma_px / 2, C.ecc / FOV], dtype=torch.float32)[None])
    def forward(s, x):
        f = F.grid_sample(s.enc(x), s.grid.expand(len(x), -1, -1, -1), align_corners=True)[..., 0].transpose(1, 2)
        h = s.mlp(torch.cat([f, s.geo.expand(len(x), -1, -1)], -1)); return h[..., 0], h[..., 1]

def run_arm(arm, C, D, K, rng, M, noise=(0., 0.), variant="both"):
    """arm in R0 | R1 | B1 | B2a | B2b -> (phosphene images (n,1,RES,RES), currents or None). M holds the trained B2 models."""
    n = len(D["y"])
    if arm == "R0": return torch.tensor(np.stack([plan_r0(D["C"][b], K, rng) for b in range(n)]), device=dev, dtype=torch.float32)[:, None], None
    if arm == "R1": I = np.stack([plan_r1(C, K, rng) for _ in range(n)])
    elif arm == "B1": I = np.stack([plan_b1(C, D["score"][b], K) for b in range(n)])
    elif arm == "B2a": I = M["tmpl"][K][D["y"]]
    else:
        with torch.no_grad(): I = currents(C, *M["b2b"][(variant, K)](D["x"]), K).cpu().numpy()
    with torch.no_grad(): P = torch.cat([C.render(torch.tensor(I[i:i + 16], device=dev), *noise) for i in range(0, n, 16)])
    return P, I

def fidelity(P, T, tau=2):
    """precision / recall / F-score of phosphene mass versus the contour, tolerance tau px (about one phosphene radius)."""
    Tn = T / T.amax((2, 3), keepdim=True).clamp_min(1e-6); Mk = (Tn > .5).float(); Dl = (F.max_pool2d(Mk, 2 * tau + 1, 1, tau) > 0).float()
    prec = ((P * Dl).sum((1, 2, 3)) / (P.sum((1, 2, 3)) + 1e-6)).cpu().numpy()
    rec = ((Mk * (F.max_pool2d((P > .05).float(), 2 * tau + 1, 1, tau))).sum((1, 2, 3)) / (Mk.sum((1, 2, 3)) + 1e-6)).cpu().numpy()
    return prec, rec, 2 * prec * rec / (prec + rec + 1e-9)

def visible(P): return np.array([ndimage.label(p[0].cpu().numpy() > .05)[1] for p in P])   # distinct phosphene blobs above a brightness criterion
def to_u8(P): return (P.clamp(0, 1) * 255).round().to(torch.uint8)


In [ ]:
# ---------------------------------------------------------------- classifiers on sparse-dot images (T_c in module B, E in module E)
def make_cnn():                                                                       # T_c: plain CNN (training classifier used inside B2b's task loss)
    blk = lambda i, o: nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2))
    return nn.Sequential(blk(1, 16), blk(16, 32), blk(32, 64), blk(64, 64), blk(64, 64), nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64, NC))

class Res(nn.Module):
    def __init__(s, i, o, st):
        super().__init__(); s.a = nn.Sequential(nn.Conv2d(i, o, 3, st, 1, bias=False), nn.BatchNorm2d(o), nn.ReLU(), nn.Conv2d(o, o, 3, 1, 1, bias=False), nn.BatchNorm2d(o))
        s.sc = nn.Sequential() if (i == o and st == 1) else nn.Sequential(nn.Conv2d(i, o, 1, st, bias=False), nn.BatchNorm2d(o))
    def forward(s, x): return F.relu(s.a(x) + s.sc(x))

def make_resnet():                                                                    # E: ResNet-style evaluator, different family from A and T_c
    return nn.Sequential(nn.Conv2d(1, 16, 3, 2, 1, bias=False), nn.BatchNorm2d(16), nn.ReLU(), Res(16, 16, 1), Res(16, 32, 2), Res(32, 64, 2), Res(64, 64, 2),
                         nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64, NC))

def front(P): return blur(P, 1.2) * 4                                                 # fixed front-end for both classifiers: dots -> small blobs

def fit_clf(net, X, y, seed, path, retrain=False):
    """Trains net on uint8 dot images X (n,1,H,W). If `path` exists the saved weights are loaded and NOTHING is trained."""
    if os.path.exists(path) and not retrain: net.load_state_dict(torch.load(path, map_location=dev)); print(f"loaded {path} (no training)"); return net.eval()
    t0 = time.time(); torch.manual_seed(seed); opt = torch.optim.Adam(net.parameters(), 2e-3); n = len(y); yt = torch.tensor(y, device=dev)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, 2e-3, total_steps=EPOCHS * ((n + 63) // 64))
    for ep in range(EPOCHS):
        net.train(); perm = torch.randperm(n, device=dev)
        for i in range(0, n, 64):
            b = perm[i:i + 64]; loss = F.cross_entropy(net(front(X[b].float() / 255)), yt[b])
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        print(f"  epoch {ep + 1}/{EPOCHS} loss {loss.item():.3f}")
    torch.save(net.state_dict(), path); print(f"trained and saved {path} ({time.time() - t0:.0f}s)"); return net.eval()

@torch.no_grad()
def clf_acc(net, X, y, bs=64):
    p = torch.cat([net(front(X[i:i + bs].float() / 255)).argmax(1).cpu() for i in range(0, len(X), bs)]); return (p.numpy() == y).astype(float)

def dot_data(C, D, mode, seed):
    """Classifier training data independent of every planner: 'board' = random dots on the true contour; 'elec' = Dynaphos renders of random contour-adjacent electrodes."""
    rng = np.random.default_rng(seed); X, y = [], []; n = len(D["y"])
    for rep in range(N_REP):
        ims = []
        for b in range(n):
            K = int(rng.integers(3, 65))
            if mode == "board":
                ys, xs = np.nonzero(D["C"][b] > .3); sel = rng.choice(len(xs), min(K, len(xs)), replace=False)
                pts = np.c_[xs[sel], ys[sel]] + rng.normal(0, 1, (len(sel), 2)); pts = pts[rng.random(len(pts)) > rng.uniform(0, .3)]
                ims.append(dots(pts, rng.uniform(1, 2.5), rng.uniform(.6, 1)))
            else:
                sc = D["score"][b] * C.usable; cand = np.flatnonzero(sc > .2 * sc.max()); sel = rng.choice(cand, min(K, len(cand)), replace=False)
                I = np.zeros(C.N, np.float32); I[sel] = rng.uniform(C.thr[sel], I_MAX); ims.append(I)
        if mode == "board": P = np.stack(ims)
        else:
            with torch.no_grad(): P = torch.cat([C.render(torch.tensor(np.stack(ims[i:i + 16]), device=dev), rng.choice([0, .1, .2]), rng.choice([0, .1, .25])) for i in range(0, n, 16)])[:, 0].cpu().numpy()
        X.append((P * 255).astype(np.uint8)); y.append(D["y"])
    return np.concatenate(X), np.concatenate(y)

def get_dots(C, D, mode, split, seed):
    """dot_data with an on-disk cache: generated once, loaded from OUT/dots_*.npz on every later run."""
    f = f"{OUT}/dots_{mode}_{split}_{seed}.npz"
    if not os.path.exists(f):
        t0 = time.time(); X, y = dot_data(C, D, mode, seed); np.savez_compressed(f, X=X, y=y); print(f"dots {mode}/{split}: generated and saved ({time.time() - t0:.0f}s)")
    z = np.load(f); return torch.tensor(z["X"], device=dev)[:, None], z["y"]


### Layout, data (cached), planners from Module B

In [ ]:
C = once("C", KEY, lambda: Phos(N_ELEC, SEED)); check_atlas(C); print(f"{C.N} electrodes, {C.usable.sum()} usable")
GB = once("GB", KEY, gabor_bank, verbose=False)
REBUILD_DATA = False                           # True = regenerate the shapes/contours once (resets itself)
if REBUILD_DATA: _mem.pop("D", None)
D = once("D", KEY, lambda: get_data(C, GB, REBUILD_DATA)); REBUILD_DATA = False
Dtr, Dva, Dte = D["train"], D["val"], D["test"]; print({k: len(v["y"]) for k, v in D.items()}, "images")

# ---------- load the frozen planners saved by the Module B notebook (nothing is trained here) ----------
def load_planners(C):
    M = {"tmpl": {}, "b2b": {}}
    for K in K_B2B:
        if os.path.exists(b2a_path(K)): M["tmpl"][K] = np.load(b2a_path(K))
        for v in VARIANTS:
            if os.path.exists(b2b_path(v, K)):
                net = Planner(C).to(dev); net.load_state_dict(torch.load(b2b_path(v, K), map_location=dev)); M["b2b"][(v, K)] = net.eval()
    assert ("both", KB) in M["b2b"], f"no planner checkpoints found in {OUT}: run the Module B notebook first (with the same settings)"
    print(f"planners loaded: templates K={sorted(M['tmpl'])}, B2b {sorted(M['b2b'])}"); return M
M = once("M", KEY, lambda: load_planners(C))
HAVE = lambda v, K=None: (v, KB if K is None else K) in M["b2b"]


### The independent evaluator

In [ ]:
# ---------- E_main: the independent evaluator. Trained once on random-dot images only (never on planner output), then saved. ----------
RETRAIN_EVALUATOR = False       # True = retrain E_main (resets itself to False afterwards)
def build_e():
    Xb, yb = get_dots(C, Dtr, "board", "train", 100 + SEED); Xe, ye = get_dots(C, Dtr, "elec", "train", 200 + SEED)
    return fit_clf(make_resnet().to(dev), torch.cat([Xb, Xe]), np.concatenate([yb, ye]), SEED, f"{OUT}/E_main_s{SEED}.pt", RETRAIN_EVALUATOR)
if RETRAIN_EVALUATOR: _mem.pop("E", None)
E = once("E", KEY, build_e); RETRAIN_EVALUATOR = False
Xbv, ybv = get_dots(C, Dva, "board", "val", 300 + SEED); Xev, yev = get_dots(C, Dva, "elec", "val", 400 + SEED)
print(f"E_main accuracy on held-out random-dot images (report this first, plan 7.1): board {clf_acc(E, Xbv, ybv).mean():.1%} | Dynaphos-elec {clf_acc(E, Xev, yev).mean():.1%}")
Tc = None
if os.path.exists(f"{OUT}/T_c_s{SEED}.pt"):
    Tc = make_cnn().to(dev); Tc.load_state_dict(torch.load(f"{OUT}/T_c_s{SEED}.pt", map_location=dev)); Tc.eval(); print("T_c loaded from Module B (used only for the trap ablation)")


In [ ]:
# ---------- helpers: bootstrap CIs, K at 50/90 %, McNemar, centre of mass; results of each experiment are cached on disk ----------
RERUN_EXPS = False              # True = recompute the experiments below even if their json exists (resets itself)
Dt, yt = Dte, Dte["y"]
def boot_ci(v, rng):
    m = [v[rng.integers(0, len(v), len(v))].mean() for _ in range(BOOT)]; return np.percentile(m, [2.5, 97.5])
def k_at(accs, ks, thr):
    for K, a in zip(ks, accs):
        if a >= thr: return K
    return None
def com_x(P): w = P.sum((1, 2)); return ((P[:, 0] * torch.arange(RES, device=P.device)[None, None]).sum((1, 2)) / (w + 1e-9)).cpu().numpy(), w.cpu().numpy()
def run_exp(name, build):
    if RERUN_EXPS: _mem.pop(name, None)
    return once(name, KEY, lambda: cached_exp(name, build))
def cached_exp(name, build):
    f = f"{OUT}/{name}_s{SEED}.json"
    if os.path.exists(f) and not RERUN_EXPS: print(f"{name}: loaded {f} (not recomputed)"); return json.load(open(f))
    t0 = time.time(); r = build(); json.dump(r, open(f, "w")); print(f"{name}: computed and saved {f} ({time.time() - t0:.0f}s)"); return r
ARMS = ["R0", "R1", "B1", "B2a", "B2b"]


### Experiment 1

In [ ]:
# ---------- Experiment 1: accuracy versus K ----------
def exp1_build():
    rng = np.random.default_rng(SEED); res = {}
    for a in ARMS:
        for K in K_LIST:
            if a == "B2a" and K not in M["tmpl"]: continue
            if a == "B2b" and not HAVE("both", K): continue
            P, I = run_arm(a, C, Dt, K, rng, M); ok = clf_acc(E, to_u8(P), yt); ci = boot_ci(ok, rng); pr, rc, fs = fidelity(P, Dt["T"])
            act = I[I > 0] if I is not None and (I > 0).any() else None
            res[f"{a}|{K}"] = dict(acc=float(ok.mean()), lo=float(ci[0]), hi=float(ci[1]), prec=float(pr.mean()), rec=float(rc.mean()), F=float(fs.mean()), vis=float(visible(P).mean()),
                                   cur_mean=float(act.mean()) if act is not None else None, cur_max=float(act.max()) if act is not None else None,
                                   charge_mA=float(I.sum(1).mean() / 1000) if I is not None else None, correct=ok.astype(int).tolist())
    return res
exp1 = run_exp("exp1", exp1_build)
print("Exp 1 | E_main top-1 accuracy [95% CI] | contour F | visible phosphenes | mean active current (uA)")
for key, r in exp1.items():
    a, K = key.split("|"); print(f"  {a:3s} K={int(K):2d}  acc {r['acc']:.1%} [{r['lo']:.1%}, {r['hi']:.1%}]  F {r['F']:.2f}  visible {r['vis']:5.1f}  I {('%.0f' % r['cur_mean']) if r['cur_mean'] else '-'}")


In [ ]:
# ---------- Figure 1, Table 1 (K at 50 / 90 %), paired McNemar tests B1 vs B2b on identical images ----------
from scipy.stats import binomtest
plt.figure(figsize=(6.5, 4.2))
for a in ARMS:
    ks = [K for K in K_LIST if f"{a}|{K}" in exp1]
    if ks:
        plt.plot(ks, [exp1[f"{a}|{K}"]["acc"] for K in ks], "o-", label=a)
        plt.fill_between(ks, [exp1[f"{a}|{K}"]["lo"] for K in ks], [exp1[f"{a}|{K}"]["hi"] for K in ks], alpha=.15)
plt.axvspan(29, 65, color="gray", alpha=.15, label="paper: 29-65 patches"); plt.axvline(5, ls=":", c="k", label="~5 phosphenes (human limit)")
plt.axhline(1 / NC, ls="--", c="r", lw=.6); plt.xlabel("active electrodes K"); plt.ylabel("E_main accuracy"); plt.legend(fontsize=7); plt.tight_layout()
plt.savefig(f"{OUT}/fig1_accuracy_vs_K_s{SEED}.png", dpi=120); plt.show()
print("Table 1: K needed for 50% / 90% accuracy (values from the K grid only)")
for a in ARMS:
    ks = [K for K in K_LIST if f"{a}|{K}" in exp1]
    if ks: accs = [exp1[f"{a}|{K}"]["acc"] for K in ks]; print(f"  {a:3s}  50%: {k_at(accs, ks, .5)}   90%: {k_at(accs, ks, .9)}")
print("\nMcNemar (exact) B1 vs B2b on the same test images")
for K in K_B2B:
    if f"B1|{K}" in exp1 and f"B2b|{K}" in exp1:
        c1, c2 = np.array(exp1[f"B1|{K}"]["correct"]), np.array(exp1[f"B2b|{K}"]["correct"]); b, c = int(((c1 == 1) & (c2 == 0)).sum()), int(((c1 == 0) & (c2 == 1)).sum())
        p = binomtest(min(b, c), b + c, .5).pvalue if b + c else 1.
        print(f"  K={K}: B1 right/B2b wrong {b}, B1 wrong/B2b right {c}, p = {p:.3f}")
print("Caution: the 29-65 band is human free naming of complex objects; our evaluator is a classifier on simple shapes. Compare as context, not like-for-like.")


### Equivariance probe

In [ ]:
# ---------- Equivariance probe: shift the input; does the phosphene centre of mass follow? ----------
def probe_build(shift=16):
    r = SIZE // RES; sh = shift // r; X = Dt["X"]
    keep = [i for i in range(len(X)) if X[i].sum() > 0 and X[i][:, -(shift + 2):].sum() == 0]          # objects that stay inside the image after the shift
    if not keep: return {}
    X0, y0 = X[keep], Dt["y"][keep]; D0 = prep_split(C, X0, y0, GB); D1 = prep_split(C, np.roll(X0, shift, axis=2), y0, GB); out = {"n": len(keep)}
    for a in ["R0", "B1", "B2a", "B2b"]:
        if a == "B2a" and KB not in M["tmpl"]: continue
        P0, _ = run_arm(a, C, D0, KB, np.random.default_rng(1), M); P1, _ = run_arm(a, C, D1, KB, np.random.default_rng(1), M)
        x0, w0 = com_x(P0); x1, w1 = com_x(P1); m = (w0 > 0) & (w1 > 0); out[a] = float(np.mean((x1 - x0)[m]) / sh)
    return out
GB = once("GB", KEY, gabor_bank, verbose=False)
probe = run_exp("probe", probe_build)
print(f"Equivariance probe (K={KB}): input shifted right by 16 px; ideal ratio = 1 (centre of mass moves by the same amount), a fixed class code gives 0")
for a, v in probe.items():
    if a != "n": print(f"  {a:3s} shift ratio {v:.2f}")


### Experiment 3: noise robustness

In [ ]:
# ---------- Experiment 3: noise robustness at K = KB (clean-trained vs robust-trained B2b, and B1) ----------
NOISES = [("p_drop", .0), ("p_drop", .05), ("p_drop", .1), ("p_drop", .2), ("p_drop", .3), ("sigma_thr", .1), ("sigma_thr", .25), ("sigma_thr", .5)]
def exp3_build():
    res = {}
    for kind, lvl in NOISES:
        nz = (lvl, 0.) if kind == "p_drop" else (0., lvl); row = {}
        for name, a, v in [("B1", "B1", "both"), ("B2b_clean", "B2b", "both"), ("B2b_robust", "B2b", "robust")]:
            if a == "B2b" and not HAVE(v): continue
            P, _ = run_arm(a, C, Dt, KB, np.random.default_rng(2), M, nz, v); row[name] = float(clf_acc(E, to_u8(P), yt).mean())
        res[f"{kind}|{lvl}"] = row
    return res
exp3 = run_exp("exp3", exp3_build)
print(f"Exp 3 | E_main accuracy at K={KB}")
for key, row in exp3.items(): print(f"  {key:16s} " + " | ".join(f"{n} {v:.1%}" for n, v in row.items()))
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
for a, kind in zip(ax, ("p_drop", "sigma_thr")):
    ks = sorted([(0. if k.startswith("p_drop|0.0") else float(k.split("|")[1]), row) for k, row in exp3.items() if k.startswith(kind) or k == "p_drop|0.0"], key=lambda t: t[0])
    for n in ("B1", "B2b_clean", "B2b_robust"):
        pts = [(l, r[n]) for l, r in ks if n in r]
        if pts: a.plot(*zip(*pts), "o-", label=n)
    a.set_xlabel(kind); a.set_ylabel("E_main accuracy"); a.legend(fontsize=7)
plt.tight_layout(); plt.savefig(f"{OUT}/fig3_robustness_s{SEED}.png", dpi=120); plt.show()


### Experiment 4: the trap ablation

In [ ]:
# ---------- Experiment 4: the trap (task loss only vs contour loss only vs both) ----------
def exp4_build():
    res = {}
    for v in ("contour_only", "task_only", "both"):
        if not HAVE(v): continue
        P, _ = run_arm("B2b", C, Dt, KB, np.random.default_rng(3), M, variant=v)
        res[v] = dict(T_c=float(clf_acc(Tc, to_u8(P), yt).mean()) if Tc is not None else None, E=float(clf_acc(E, to_u8(P), yt).mean()), F=float(fidelity(P, Dt["T"])[2].mean()))
    return res
exp4 = run_exp("exp4", exp4_build)
print(f"Exp 4 | trap ablation at K={KB}: training classifier T_c vs independent evaluator E vs contour F-score")
for v, r in exp4.items(): print(f"  {v:13s} T_c {('%.1f%%' % (100 * r['T_c'])) if r['T_c'] is not None else '-'} | E {r['E']:.1%} | F {r['F']:.2f}")
if len(exp4) > 1:
    fig, ax = plt.subplots(figsize=(6, 3.4)); w = .27; xs = np.arange(len(exp4))
    for i, (k, lab) in enumerate((("T_c", "training classifier T_c"), ("E", "independent evaluator E"), ("F", "contour F-score"))):
        ax.bar(xs + (i - 1) * w, [(r[k] or 0) for r in exp4.values()], w, label=lab)
    ax.set_xticks(xs); ax.set_xticklabels(list(exp4)); ax.legend(fontsize=7); plt.tight_layout(); plt.savefig(f"{OUT}/fig4_trap_s{SEED}.png", dpi=120); plt.show()
RERUN_EXPS = False              # reset: the experiments above are cached again from now on
